# OTD workbook validation (read-only)

Inspect the 2025 and 2026 workbook sheets, identify delivery-data tabs, and preview the normalized OTD rows and site coverage. This notebook does not connect to or write to SQL.

The source worksheet names are explicitly configured through `GC2S (2)` in 2025 and `GC2S` in 2026. Consolidated, graph, and report tabs are excluded. The inventory still shows all workbook sheets.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

FILE_2025 = "data/OTD_data_2025.xlsx"
FILE_2026 = "data/otd_data.xlsx"

# Source worksheets only; summary and report tabs are excluded.
SHEET_NAMES_2025 = [
    "AS", "AW", "MP", "PSC", "MR&PS", "NGA", "WIMS", "GC2S (2)",
]
SHEET_NAMES_2026 = [
    "SDS", "AS", "AW", "MP", "PSC", "MP - Chris",
    "MR&PS", "NGA", "WIMS", "GC2S",
]

# Optional exception if Program is in column B on a specific sheet.
SECOND_COLUMN_PROGRAM_SHEET = "WIMS"


## Workbook normalization

The sheet detector looks for delivery timeline values and monthly columns, without relying on facility or worksheet names. The existing loader handles identifier fill-down, commitment selection, and 2025/2026 alignment. Enrichment only fills missing 2025 Project ID, Site, or Type fields when the 2026 value is unambiguous; all existing values and original business-unit labels are preserved.


In [ ]:
MONTHS = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]

FINAL_COLUMNS = [
    "Timeline",
    "Program",
    "BU",
    "Project ID",
    "Site",
    "Type",
    *MONTHS,
    "Total",
    "Year",
]

TIMELINE_VALUES = {
    "contract commitment",
    "factory planned",
    "planned",
    "actual",
    "actual delivered",
    "actuals delivered",
}

TIMELINE_RENAMES = {
    "contract commitment": "Contract Commitment",
    "factory planned": "Contract Commitment",
    "planned": "Contract Commitment",
    "actual": "Actuals Delivered",
    "actual delivered": "Actuals Delivered",
    "actuals delivered": "Actuals Delivered",
}

MONTH_RENAMES = {
    "JAN": "Jan",
    "FEB": "Feb",
    "MAR": "Mar",
    "APR": "Apr",
    "APRIL": "Apr",
    "MAY": "May",
    "JUN": "Jun",
    "JUL": "Jul",
    "AUG": "Aug",
    "SEP": "Sep",
    "SEPT": "Sep",
    "OCT": "Oct",
    "NOV": "Nov",
    "DEC": "Dec",
}


def normalize_header(value):
    return str(value).strip().lower().replace("_", " ")


def detect_timeline_column(df, sheet_name):
    for column in df.columns:
        if normalize_header(column) == "timeline":
            return column

    best_column = None
    best_score = 0

    for column in df.columns:
        values = df[column].dropna().astype(str).str.strip().str.lower()
        score = values.isin(TIMELINE_VALUES).sum()

        if score > best_score:
            best_column = column
            best_score = score

    if best_column is None or best_score == 0:
        raise ValueError(
            f"Could not identify the timeline column on sheet {sheet_name!r}. "
            f"Columns found: {list(df.columns)}"
        )

    return best_column


def normalize_identifier_columns(df):
    rename_map = {}

    for column in df.columns:
        key = normalize_header(column)

        if key in {"project", "project id"}:
            rename_map[column] = "Project ID"
        elif key == "site":
            rename_map[column] = "Site"
        elif key == "type":
            rename_map[column] = "Type"

    return df.rename(columns=rename_map)


def normalize_month_columns(df):
    rename_map = {}

    for column in df.columns:
        month_name = MONTH_RENAMES.get(str(column).strip().upper())
        if month_name:
            rename_map[column] = month_name

    return df.rename(columns=rename_map)


def align_otd(df):
    df = df.copy()

    for column in ["Project ID", "Site", "Type"]:
        if column not in df.columns:
            df[column] = pd.NA

    for month in MONTHS:
        if month not in df.columns:
            df[month] = 0
        df[month] = pd.to_numeric(df[month], errors="coerce").fillna(0)

    df["Total"] = df[MONTHS].sum(axis=1)

    for column in FINAL_COLUMNS:
        if column not in df.columns:
            df[column] = pd.NA

    return df[FINAL_COLUMNS]


def select_commitment_rows(df):
    timeline = df["Timeline"].astype(str).str.strip().str.lower()
    identity_columns = [
        column for column in ["Program", "Project ID", "Site", "Type"]
        if column in df.columns
    ]
    group_keys = df[identity_columns].copy()

    for column in identity_columns:
        group_keys[column] = (
            group_keys[column].fillna("").astype(str).str.strip()
        )

    is_contract = timeline.eq("contract commitment")
    is_factory_planned = timeline.eq("factory planned")
    is_planned = timeline.eq("planned")
    contract_has_values = is_contract & df[MONTHS].ne(0).any(axis=1)

    selection = group_keys.copy()
    selection["_contract_has_values"] = contract_has_values.to_numpy()
    selection["_has_factory_planned"] = is_factory_planned.to_numpy()

    grouped = selection.groupby(identity_columns, dropna=False)
    group_has_contract_values = grouped["_contract_has_values"].transform("any")
    group_has_factory_planned = grouped["_has_factory_planned"].transform("any")

    is_commitment_candidate = is_contract | is_factory_planned | is_planned
    keep_commitment = (
        contract_has_values
        | (is_factory_planned & ~group_has_contract_values)
        | (is_planned & ~group_has_contract_values & ~group_has_factory_planned)
    )

    return df[~is_commitment_candidate | keep_commitment].copy()


def load_otd_sheet(path, sheet_name, year):
    df = pd.read_excel(path, sheet_name=sheet_name).dropna(how="all")

    comment_columns = [
        column for column in df.columns
        if normalize_header(column) == "comments"
    ]
    empty_helper_columns = [
        column for column in df.columns
        if "unnamed" in str(column).lower() and df[column].isna().all()
    ]
    df = df.drop(columns=comment_columns + empty_helper_columns, errors="ignore")

    if year == 2025:
        total_rows = df.astype("string").apply(
            lambda column: column.str.contains(
                "total", case=False, na=False, regex=False
            )
        ).any(axis=1)
        dropped_total_rows = int(total_rows.sum())
        df = df[~total_rows].copy()

        if dropped_total_rows:
            print(
                f"2025 / {sheet_name}: dropped "
                f"{dropped_total_rows:,} row(s) containing 'Total'"
            )

    if len(df.columns) == 0:
        raise ValueError(f"Sheet {sheet_name!r} has no usable columns.")

    program_column_index = (
        1 if year == 2026 and sheet_name == SECOND_COLUMN_PROGRAM_SHEET else 0
    )
    if len(df.columns) <= program_column_index:
        raise ValueError(
            f"Sheet {sheet_name!r} does not have column index "
            f"{program_column_index} for Program."
        )

    program_column = df.columns[program_column_index]
    other_program_columns = [
        column for index, column in enumerate(df.columns)
        if index != program_column_index and normalize_header(column) == "program"
    ]
    df = df.drop(columns=other_program_columns, errors="ignore")
    df.rename(columns={program_column: "Program"}, inplace=True)

    timeline_column = detect_timeline_column(df, sheet_name)
    if timeline_column != "Timeline":
        df.rename(columns={timeline_column: "Timeline"}, inplace=True)

    df = normalize_identifier_columns(df)
    df = normalize_month_columns(df)

    for column in ["Program", "Project ID", "Site", "Type"]:
        if column in df.columns:
            df[column] = df[column].ffill()

    for month in MONTHS:
        if month not in df.columns:
            df[month] = 0
        df[month] = pd.to_numeric(df[month], errors="coerce").fillna(0)

    df = select_commitment_rows(df)
    timeline = df["Timeline"].astype(str).str.strip().str.lower()
    df["Timeline"] = timeline.replace(TIMELINE_RENAMES)
    df = df[
        df["Timeline"].isin(["Contract Commitment", "Actuals Delivered"])
    ].copy()

    df["BU"] = sheet_name
    df["Year"] = year

    df = df[df["Program"].notna()]
    df = df[df["Program"].astype(str).str.strip().ne("")]

    return align_otd(df.reset_index(drop=True))


def load_otd_year(path, sheet_names, year):
    if not sheet_names:
        raise ValueError(f"Add the {year} sheet names to SHEET_NAMES_{year}.")

    frames = []

    for sheet_name in sheet_names:
        frame = load_otd_sheet(
            path,
            sheet_name=sheet_name,
            year=year,
        )
        print(f"{year} / {sheet_name}: {len(frame):,} normalized rows")
        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


def normalize_program_match_key(value):
    if pd.isna(value):
        return ""

    return "".join(
        character.casefold()
        for character in str(value)
        if character.isalnum()
    )


def enrich_2025_details_from_2026(otd_2025, otd_2026):
    # Business unit belongs to its original worksheet and is never enriched.
    detail_columns = ["Project ID", "Site", "Type"]
    lookup_source = otd_2026[
        otd_2026["Program"].notna()
    ][["Program", *detail_columns]].copy()
    lookup_source["_program_match_key"] = (
        lookup_source["Program"].map(normalize_program_match_key)
    )
    lookup_source = lookup_source[
        lookup_source["_program_match_key"].ne("")
    ].copy()

    for column in detail_columns:
        has_value = (
            lookup_source[column].notna()
            & lookup_source[column].astype(str).str.strip().ne("")
        )
        lookup_source[column] = lookup_source[column].where(has_value, pd.NA)

    value_counts = (
        lookup_source.groupby("_program_match_key", sort=False)[detail_columns]
        .nunique(dropna=True)
    )
    conflicting_field_count = int((value_counts > 1).sum().sum())

    def consensus_value(values):
        populated = values.dropna().drop_duplicates()
        return populated.iloc[0] if len(populated) == 1 else pd.NA

    lookup = (
        lookup_source.groupby("_program_match_key", sort=False, as_index=False)
        .agg({column: consensus_value for column in detail_columns})
    )
    enriched = otd_2025.copy()
    enriched["_program_match_key"] = (
        enriched["Program"].map(normalize_program_match_key)
    )
    enriched = enriched.merge(
        lookup,
        on="_program_match_key",
        how="left",
        suffixes=("", "_2026"),
        validate="many_to_one",
        indicator=True,
    )
    matched_program_count = enriched.loc[
        enriched["_merge"].eq("both"), "_program_match_key"
    ].nunique()

    filled_counts = {}
    for column in detail_columns:
        source_column = f"{column}_2026"
        existing = enriched[column]
        has_existing = (
            existing.notna()
            & existing.astype(str).str.strip().ne("")
        )
        missing_before = ~has_existing
        # Fill only missing 2025 values using unambiguous 2026 details.
        enriched[column] = existing.where(has_existing, enriched[source_column])
        filled_counts[column] = int(
            (missing_before & enriched[column].notna()).sum()
        )
        enriched.drop(columns=source_column, inplace=True)

    enriched.drop(columns=["_merge", "_program_match_key"], inplace=True)
    # A cross-year match must never reassign the source worksheet's BU.
    if not enriched["BU"].reset_index(drop=True).equals(
        otd_2025["BU"].reset_index(drop=True)
    ):
        raise ValueError("2025 business-unit values changed during enrichment.")
    print(
        f"Matched 2025 rows to {matched_program_count:,} "
        "alphanumeric Program key(s) from 2026"
    )
    if conflicting_field_count:
        print(
            f"Skipped {conflicting_field_count:,} conflicting "
            "2026 Program/detail combination(s)"
        )
    print(
        "Filled missing 2025 values from 2026: "
        + ", ".join(f"{column}={filled_counts[column]:,}" for column in detail_columns)
    )

    return enriched[FINAL_COLUMNS]


## Sheet inventory and normalized coverage

First, review which sheets are included or excluded. Then review their normalized row and site counts. Sheet-level processing errors are reported without stopping the other sheets.


In [ ]:
def inspect_workbook(path, year, configured_names):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Workbook not found: {path}")
    results = []
    with pd.ExcelFile(path) as book:
        unknown = sorted(set(configured_names) - set(book.sheet_names))
        if unknown:
            raise ValueError(f"Unknown {year} worksheet names: {unknown}")
        for sheet in book.sheet_names:
            candidate, note, site_samples = False, "", ""
            try:
                df = pd.read_excel(book, sheet_name=sheet, nrows=150).dropna(how="all")
                month_count = len({MONTH_RENAMES.get(str(c).strip().upper()) for c in df.columns} - {None})
                try:
                    timeline_col = detect_timeline_column(df, sheet)
                    values = df[timeline_col].dropna().astype(str).str.strip().str.lower()
                    has_commitment = values.isin(["contract commitment", "factory planned", "planned"]).any()
                    has_actuals = values.isin(["actual", "actual delivered", "actuals delivered"]).any()
                    candidate = month_count >= 2 and has_commitment and has_actuals
                except ValueError:
                    pass
                site_cols = [c for c in df.columns if normalize_header(c) == "site"]
                if site_cols:
                    site_values = df[site_cols[0]].dropna().astype(str).str.strip()
                    site_samples = ", ".join(x for x in site_values.unique()[:3] if x)
                    if not site_samples:
                        note = "Site column is empty"
                else:
                    note = "No Site column"
            except Exception as exc:
                note = f"Inspection error: {str(exc)[:100]}"
            selected = sheet in configured_names if configured_names else candidate
            status = "Included" if selected else "Excluded candidate" if candidate else "Skipped"
            results.append({"Year": year, "Sheet": sheet, "Status": status,
                            "Example Sites": site_samples, "Note": note,
                            "_selected": selected})
    report = pd.DataFrame(results)
    return report, report.loc[report["_selected"], "Sheet"].tolist()


def validate_sheets(path, year, names):
    frames, notes = [], []
    for sheet in names:
        try:
            frame = load_otd_sheet(path, sheet, year)
            site = frame["Site"].fillna("").astype(str).str.strip()
            missing = site.eq("")
            notes.append({"Year": year, "Sheet": sheet, "Status": "Loaded",
                          "Rows": len(frame), "Programs": frame["Program"].nunique(),
                          "Distinct Sites": site[~missing].nunique(),
                          "Missing Site Rows": int(missing.sum()),
                          "Example Sites": ", ".join(site[~missing].unique()[:4])})
            frames.append(frame)
        except Exception as exc:
            notes.append({"Year": year, "Sheet": sheet,
                          "Status": f"ERROR: {str(exc)[:120]}",
                          "Rows": 0, "Programs": 0, "Distinct Sites": 0,
                          "Missing Site Rows": 0, "Example Sites": ""})
    data = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=FINAL_COLUMNS)
    return data, pd.DataFrame(notes)


inventory_2025, names_2025 = inspect_workbook(FILE_2025, 2025, SHEET_NAMES_2025)
inventory_2026, names_2026 = inspect_workbook(FILE_2026, 2026, SHEET_NAMES_2026)
print("1. Workbook sheets (including exclusions):")
display(pd.concat([inventory_2025, inventory_2026], ignore_index=True).drop(columns="_selected"))

otd_2025, coverage_2025 = validate_sheets(FILE_2025, 2025, names_2025)
otd_2026, coverage_2026 = validate_sheets(FILE_2026, 2026, names_2026)
print("2. Loaded sheet coverage:")
display(pd.concat([coverage_2025, coverage_2026], ignore_index=True))

if otd_2025.empty and otd_2026.empty:
    raise ValueError("No OTD rows loaded. Inspect the worksheet inventory above.")
if not otd_2025.empty and not otd_2026.empty:
    otd_2025 = enrich_2025_details_from_2026(otd_2025, otd_2026)
otd = pd.concat([otd_2025, otd_2026], ignore_index=True)
print(f"Total normalized rows: {len(otd):,}")


## Normalized OTD inventory

Review the resulting year, worksheet, and site values. This previews the in-memory normalized dataset; nothing is uploaded.


In [ ]:
normalized = otd.assign(SiteLabel=otd["Site"].fillna("").astype(str).str.strip().replace("", "(missing)"))
summary = (normalized.groupby(["Year", "BU"], dropna=False)
           .agg(Rows=("Program", "size"), Programs=("Program", "nunique"),
                Sites=("SiteLabel", lambda x: x[x != "(missing)"].nunique()),
                MissingSiteRows=("SiteLabel", lambda x: (x == "(missing)").sum()))
           .reset_index().sort_values(["Year", "BU"], ascending=[False, True]))
print("3. Final year and BU coverage:")
display(summary)

sites = (normalized.groupby(["Year", "BU", "SiteLabel"], dropna=False)
         .size().rename("Rows").reset_index()
         .sort_values(["Year", "BU", "Rows"], ascending=[False, True, False]))
print("4. Normalized sites (first 40):")
display(sites.head(40))
print(f"Total site combinations: {len(sites):,}")


## Validation only

This notebook contains no SQL connection or database upload step. Updates to SQL require a separate, explicitly reviewed process.
